# 7 Wonders: symulacja i trening ONNX

Ten notebook uruchamia symulację w C#, zbiera plik wynikowy z logami ruchów i trenuje hierarchiczny model policy/value na faktycznym wektorze stanu.

Schemat treningu PUCT + value:
1. Self-play na agencie `puct` zapisuje `MoveLog.PolicyTarget` z visit counts korzenia.
2. `value_target` jest liczony z wyniku końcowego gry.
3. Model uczy się soft cross-entropy dla policy i MSE dla value.
4. Po treningu eksportujesz `policy_network.onnx` i podpinasz go do nowego agenta `puct`.

Przepływ:
1. `GameConsole` eksportuje dane z `MoveLog.State`, `MoveLog.ActionMask`, `MoveLog.ActionIndex` i opcjonalnie `MoveLog.PolicyTarget`.
2. Notebook wczytuje najnowszy plik `training_*.json`.
3. Model PyTorch uczy się i eksportuje `policy_network.onnx`.

In [1]:
import os

EPOCHS = int(os.getenv("WONDERS_EPOCHS", "50"))
GAMES_TRAIN = int(os.getenv("WONDERS_GAMES", "100"))
SEED = int(os.getenv("WONDERS_SEED", "1"))
OUTPUT_NAME = os.getenv("WONDERS_OUTPUT_NAME", "self_play_puct")
MINIMAL_LOGS = os.getenv("WONDERS_MINIMAL_LOGS", "1").lower() not in {"0", "false", "no"}
BATCH_SIZE = int(os.getenv("WONDERS_BATCH_SIZE", "32"))

In [2]:
from pathlib import Path
import importlib
import subprocess
import sys
import os

repo_root = Path(r"c:/Users/kubeu/Kuba-dokumenty/Magisterka/7 Wonders")
game_console = repo_root / "GameConsole" / "GameConsole.csproj"
results_dir = repo_root / "GameConsole" / "Results"
encoding_dir = repo_root / "GameAI" / "Encoding"
model_path = Path(os.getenv("WONDERS_MODEL_PATH", str(encoding_dir / "onnx_models" / "policy_network_50_200.onnx")))

sys.path.append(str(encoding_dir))

import game_training_pipeline as gtp
gtp = importlib.reload(gtp)

ActionSpace = gtp.ActionSpace
GameDataset = gtp.GameDataset
HierarchicalPolicyNetwork = gtp.HierarchicalPolicyNetwork
train_epoch = gtp.train_epoch
evaluate = gtp.evaluate

print("Repo root:", repo_root)
print("State vector size:", ActionSpace.STATE_VECTOR_SIZE)
print("Primary action size:", ActionSpace.TOTAL_PRIMARY_ACTIONS)
print("Model path:", model_path)

Repo root: c:\Users\kubeu\Kuba-dokumenty\Magisterka\7 Wonders
State vector size: 1903
Primary action size: 120
Model path: c:\Users\kubeu\Kuba-dokumenty\Magisterka\7 Wonders\GameAI\Encoding\onnx_models\policy_network_50_200.onnx


In [3]:
import torch
print(f"Czy CUDA działa? {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Wykryta karta: {torch.cuda.get_device_name(0)}")
    print(f"Wersja CUDA w Torch: {torch.version.cuda}") # type: ignore

Czy CUDA działa? True
Wykryta karta: NVIDIA GeForce RTX 4050 Laptop GPU
Wersja CUDA w Torch: 12.4


In [ ]:
def run_self_play_training(
    seed: int = SEED,
    games: int = GAMES_TRAIN,
    agent1: str = "puct",
    agent2: str = "puct",
    output_name: str = OUTPUT_NAME,
    minimal_logs: bool = MINIMAL_LOGS,
    model: Path = model_path,
):
    results_dir.mkdir(parents=True, exist_ok=True)
    command = [
        "dotnet", "run",
        "--project", str(game_console),
        "--",
        "self-play-train",
        "--seed", str(seed),
        "--games", str(games),
        "--model", str(model),
        "--output", output_name,
    ]
    if minimal_logs:
        command.append("--minimal-logs")
    else:
        command.append("--full-logs")
    subprocess.run(command, cwd=repo_root, check=True)

    file_name = f"{output_name}_{games}_games{'_minimal' if minimal_logs else ''}.json"
    return results_dir / file_name

training_file = run_self_play_training(seed=SEED, games=GAMES_TRAIN, output_name=OUTPUT_NAME, minimal_logs=MINIMAL_LOGS)
print("Simulation finished:", training_file)

## Automatyczna runda self-play + trening

Ten notebook może działać w dwóch trybach:
- interaktywnie, gdy uruchamiasz pojedyncze komórki,
- automatycznie, gdy odpalasz go z `run_self_play_train.ps1`.

W trybie automatycznym notebook:
1. uruchamia self-play PUCT,
2. zapisuje plik `self_play_puct_*.json`,
3. wczytuje go do `GameDataset`,
4. trenuje `HierarchicalPolicyNetwork`,
5. eksportuje `policy_network_*.onnx`.

Zmienne środowiskowe sterujące rundą:
- `WONDERS_SEED`
- `WONDERS_GAMES`
- `WONDERS_EPOCHS`
- `WONDERS_OUTPUT_NAME`
- `WONDERS_MODEL_PATH`
- `WONDERS_MINIMAL_LOGS`

Jeśli chcesz tylko samą symulację, dalej możesz uruchamiać ją z poziomu C# bez notebooka.

In [5]:
from torch.utils.data import DataLoader, Subset
import random
import copy
import torch

candidate_files = []
if 'training_file' in globals() and training_file:
    candidate_files.append(Path(training_file))
candidate_files.extend(sorted(results_dir.glob(f"{OUTPUT_NAME}_*_games*.json")))
candidate_files.extend(sorted(results_dir.glob("training_*.json")))

existing_files = [path for path in candidate_files if path.exists()]
if not existing_files:
    raise FileNotFoundError(f"No training files found in {results_dir}")

latest_file = max(existing_files, key=lambda path: path.stat().st_mtime)
print("Using dataset:", latest_file)

dataset = GameDataset(str(latest_file), normalize=True, validate_shapes=True)

indices_by_game = {}
for index, sample in enumerate(dataset.data):
    match_id = sample.get('match_id') or f'fallback_{index}'
    indices_by_game.setdefault(match_id, []).append(index)

game_ids = list(indices_by_game.keys())
rng = random.Random(42)
rng.shuffle(game_ids)
if len(game_ids) <= 1:
    train_game_ids = game_ids
    val_game_ids = []
else:
    train_game_count = max(1, int(len(game_ids) * 0.6))
    if train_game_count >= len(game_ids):
        train_game_count = len(game_ids) - 1
    train_game_ids = game_ids[:train_game_count]
    val_game_ids = game_ids[train_game_count:]

train_indices = [idx for game_id in train_game_ids for idx in indices_by_game[game_id]]
val_indices = [idx for game_id in val_game_ids for idx in indices_by_game[game_id]]

print(f"Games: train={len(train_game_ids)} val={len(val_game_ids)}")
print(f"Samples: train={len(train_indices)} val={len(val_indices)}")

train_dataset = Subset(dataset, train_indices)
val_dataset = Subset(dataset, val_indices) if val_indices else None
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False) if val_dataset is not None else None

# Hyperparameters (can be overridden via environment variables)
LR = float(os.getenv("WONDERS_LR", "1e-4"))
VALUE_LOSS_WEIGHT = float(os.getenv("WONDERS_VALUE_LOSS_WEIGHT", "0.25"))
PATIENCE = int(os.getenv("WONDERS_PATIENCE", "25"))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = HierarchicalPolicyNetwork(state_dim=ActionSpace.STATE_VECTOR_SIZE, hidden_dim=256, dropout=0.1).to(device)
# Optimizer with configurable LR and weight decay
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
# Scheduler to anneal LR across epochs
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

# Quick sanity checks before training
with torch.no_grad():
    if val_loader is not None and len(val_loader) > 0:
        print("Untrained val total_loss:", evaluate(model, val_loader, device, value_weight=VALUE_LOSS_WEIGHT)['total_loss'])

best_val_loss = float("inf")
best_epoch = 0
best_state_dict = None
no_improve = 0
patience = PATIENCE  # early stopping patience (epochs)

for epoch in range(EPOCHS):
    train_stats = train_epoch(model, train_loader, optimizer, device, value_weight=VALUE_LOSS_WEIGHT)
    if val_loader is not None:
        val_stats = evaluate(model, val_loader, device, value_weight=VALUE_LOSS_WEIGHT)
        if val_stats["total_loss"] < best_val_loss - 1e-8:
            best_val_loss = val_stats["total_loss"]
            best_epoch = epoch + 1
            best_state_dict = copy.deepcopy(model.state_dict())
            no_improve = 0
        else:
            no_improve += 1
        print(f"epoch={epoch + 1} train={train_stats['total_loss']:.4f} val={val_stats['total_loss']:.4f} best={best_val_loss:.4f}@{best_epoch}")
    else:
        print(f"epoch={epoch + 1} train={train_stats['total_loss']:.4f}")
        if epoch == 0:
            best_state_dict = copy.deepcopy(model.state_dict())
            best_epoch = 1

    # Step LR scheduler after each epoch
    try:
        scheduler.step()
    except Exception:
        pass

    # Early stopping check
    if val_loader is not None and no_improve >= patience:
        print(f"Early stopping at epoch {epoch+1} (no improvement for {no_improve} epochs)")
        break

if best_state_dict is not None:
    model.load_state_dict(best_state_dict)
    print(f"Restored best checkpoint from epoch {best_epoch}")

2026-05-18 17:29:44,089 - game_training_pipeline - INFO - Loading dataset from c:\Users\kubeu\Kuba-dokumenty\Magisterka\7 Wonders\GameConsole\Results\self_play_puct_20260518_144032_100_games_minimal.json


Using dataset: c:\Users\kubeu\Kuba-dokumenty\Magisterka\7 Wonders\GameConsole\Results\self_play_puct_20260518_144032_100_games_minimal.json


2026-05-18 17:29:46,055 - game_training_pipeline - INFO - Loaded 5772 valid samples
2026-05-18 17:29:46,120 - game_training_pipeline - INFO - State normalized: mean=0.0477, std=0.1095


Games: train=60 val=40
Samples: train=3457 val=2315
Untrained val total_loss: 1.7332113875168704
epoch=1 train=1.6439 val=1.6303 best=1.6303@1
epoch=2 train=1.4759 val=1.6852 best=1.6303@1
epoch=3 train=1.4136 val=1.7232 best=1.6303@1
epoch=4 train=1.3880 val=1.7382 best=1.6303@1
epoch=5 train=1.3733 val=1.7465 best=1.6303@1
epoch=6 train=1.3622 val=1.7511 best=1.6303@1
epoch=7 train=1.3539 val=1.7523 best=1.6303@1
epoch=8 train=1.3496 val=1.7467 best=1.6303@1
epoch=9 train=1.3450 val=1.7475 best=1.6303@1
epoch=10 train=1.3406 val=1.7476 best=1.6303@1
epoch=11 train=1.3382 val=1.7493 best=1.6303@1
epoch=12 train=1.3353 val=1.7471 best=1.6303@1
epoch=13 train=1.3332 val=1.7480 best=1.6303@1
epoch=14 train=1.3314 val=1.7449 best=1.6303@1
epoch=15 train=1.3301 val=1.7420 best=1.6303@1
epoch=16 train=1.3292 val=1.7446 best=1.6303@1
epoch=17 train=1.3281 val=1.7489 best=1.6303@1
epoch=18 train=1.3273 val=1.7459 best=1.6303@1
epoch=19 train=1.3268 val=1.7517 best=1.6303@1
epoch=20 train=1.32

In [5]:
onnx_path = encoding_dir / f"onnx_models/puct_{EPOCHS}_{GAMES_TRAIN}_match_split.onnx"
model.onnx_export(str(onnx_path), validate=True)
print("Exported:", onnx_path)
print(f"Best epoch used for export: {best_epoch}")

batch = next(iter(train_loader))
state = batch['state'].to(device)
action_mask = batch['action_mask'].to(device)
outputs = model(state, action_mask=action_mask)
print("policy shape:", outputs['policy_masked_logits'].shape)
print("value shape:", outputs['value'].shape)
print("Training round finished.")

2026-05-18 14:06:02,351 - game_training_pipeline - INFO - Exporting model to ONNX format: c:\Users\kubeu\Kuba-dokumenty\Magisterka\7 Wonders\GameAI\Encoding\onnx_models\puct_50_100_match_split.onnx
c:\Users\kubeu\Kuba-dokumenty\Magisterka\7 Wonders\GameAI\Encoding\game_training_pipeline.py:140: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  assert action_mask.shape == policy_logits.shape, \
2026-05-18 14:06:02,474 - game_training_pipeline - INFO - ✓ Model exported successfully: c:\Users\kubeu\Kuba-dokumenty\Magisterka\7 Wonders\GameAI\Encoding\onnx_models\puct_50_100_match_split.onnx
2026-05-18 14:06:02,474 - game_training_pipeline - INFO - Validating ONNX model: c:\Users\kubeu\Kuba-dokumenty\Magisterka\7 Wonders\GameAI\Encoding\onnx_models\puct_50_100_match_split.onnx


Exported: c:\Users\kubeu\Kuba-dokumenty\Magisterka\7 Wonders\GameAI\Encoding\onnx_models\puct_50_100_match_split.onnx
Best epoch used for export: 4
policy shape: torch.Size([32, 120])
value shape: torch.Size([32, 1])
Training round finished.
